# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 2: Data Collection with Web Scraping

This notebook collects historical Falcon 9 launch records from the Wikipedia page *List of Falcon 9 and Falcon Heavy launches*.

**Objectives**
- Request the Wikipedia page
- Extract the column names from the HTML table header
- Parse the launch tables into a Pandas dataframe

In [1]:
import re
import unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup

### Helper functions for processing HTML table cells

In [2]:
def date_time(table_cells):
    """Return the date and time from a table cell."""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """Return the booster version from a table cell."""
    out = ''.join([bv for i, bv in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """Return the landing status from a table cell."""
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    """Return the payload mass (text up to and including 'kg')."""
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    """Return the column name from a table header cell."""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not column_name.strip().isdigit():
        column_name = column_name.strip()
        return column_name

def cell_text(cell):
    """Text of the first link in a cell, or the plain cell text when there is no link."""
    if cell.a and cell.a.string:
        return cell.a.string.strip()
    return cell.get_text(strip=True)

### Task 1: Request the Falcon 9 launch Wikipedia page
A fixed snapshot of the page (June 2021) is used so the results stay consistent.

In [3]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# Wikipedia rejects requests that do not identify themselves, so send a User-Agent header
headers = {"User-Agent": "Mozilla/5.0 (educational project; SpaceX capstone data collection)"}

response = requests.get(static_url, headers=headers, timeout=60)
response.raise_for_status()
print("Status code:", response.status_code)

Status code: 200


In [4]:
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.title.string)

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### Task 2: Extract all column names from the HTML table header

In [5]:
html_tables = soup.find_all('table')
print("Number of tables on the page:", len(html_tables))

# The third table is the first launch records table
first_launch_table = html_tables[2]

column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

Number of tables on the page: 29
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


### Task 3: Create a dataframe by parsing the launch HTML tables

In [6]:
launch_dict = {'Flight No.': [],
               'Launch site': [],
               'Payload': [],
               'Payload mass': [],
               'Orbit': [],
               'Customer': [],
               'Launch outcome': [],
               'Version Booster': [],
               'Booster landing': [],
               'Date': [],
               'Time': []}

In [7]:
extracted_row = 0

# Loop over every launch table on the page
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        # A launch row starts with a header cell holding the flight number
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
            else:
                flag = False
        else:
            flag = False

        row = rows.find_all('td')
        if flag and len(row) >= 9:
            extracted_row += 1

            # Flight number
            launch_dict['Flight No.'].append(flight_number)

            # Date and time
            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1] if len(datatimelist) > 1 else None)

            # Booster version
            bv = booster_version(row[1])
            if not bv:
                bv = cell_text(row[1])
            launch_dict['Version Booster'].append(bv)

            # Launch site, payload, payload mass, orbit, customer
            launch_dict['Launch site'].append(cell_text(row[2]))
            launch_dict['Payload'].append(cell_text(row[3]))
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(cell_text(row[5]))
            launch_dict['Customer'].append(cell_text(row[6]))

            # Launch outcome and booster landing
            launch_dict['Launch outcome'].append(list(row[7].strings)[0].strip())
            launch_dict['Booster landing'].append(landing_status(row[8]).strip())

print("Launch records extracted:", extracted_row)

Launch records extracted: 121


In [8]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
print(df.shape)
df.head(10)

(121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10
5,6,VAFB,CASSIOPE,500 kg,Polar orbit,MDA,Success,F9 v1.17B1003,Uncontrolled,29 September 2013,16:00
6,7,CCAFS,SES-8,"3,170 kg",GTO,SES,Success,F9 v1.1,No,3 December 2013,22:41
7,8,CCAFS,Thaicom 6,"3,325 kg",GTO,Thaicom,Success,F9 v1.1,No,6 January 2014,22:06
8,9,Cape Canaveral,SpaceX CRS-3,"2,296 kg",LEO,NASA,Success,F9 v1.1,Controlled,18 April 2014,19:25
9,10,Cape Canaveral,Orbcomm-OG2,"1,316 kg",LEO,Orbcomm,Success,F9 v1.1,Controlled,14 July 2014,15:15


In [9]:
df['Launch site'].value_counts()

,count
Launch site,
CCAFS,40
KSC,33
Cape Canaveral,20
VAFB,16
CCSFS,12


### Export the result

In [10]:
df.to_csv('spacex_web_scraped.csv', index=False)
print("Saved spacex_web_scraped.csv with", df.shape[0], "rows")

Saved spacex_web_scraped.csv with 121 rows
